# VirulentHunter benchmark

In [1]:
# library
from pathlib import Path
from tqdm import tqdm

import numpy as np
import pandas as pd
import torch

import os
import src

tqdm.pandas()

# directory
root = Path.cwd()
base_dir = os.path.join(root, "data", "VirulentHunter")
os.chdir(base_dir)

/home/sjchoi/miniconda3/envs/node09/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def preprocess(fasta_path, label_path, desc_path, tax_path, id_field="id", replace=True):
    df = src.fasta_to_df(fasta_path)
    label = pd.read_csv(label_path).drop(columns=["Unnamed: 0"])  # drop index
    df = df.merge(label)

    # interproscan
    scan_df = pd.read_csv(desc_path)
    df = df.merge(scan_df)

    # taxonomy
    tax_df = pd.read_csv(tax_path)
    df = df.merge(tax_df)

    if replace:
        mask = df["sequence"].str.contains("J|\\*", regex=True, na=False)
        print("rows with J or *:", mask.sum())
        df.loc[mask, "sequence"] = df.loc[mask, "sequence"].str.replace("J|\\*", "X", regex=True)
    
    df = df.drop_duplicates(subset=["id"], keep="first").reset_index(drop=True)

    return df

train = preprocess("train.fasta", "train_labels.csv", "df_interproscan.csv", "df_taxonomy_gtdb.csv")
val   = preprocess("val.fasta",   "val_labels.csv", "df_interproscan.csv", "df_taxonomy_gtdb.csv")
test  = preprocess("test.fasta",  "test_labels.csv", "df_interproscan.csv", "df_taxonomy_gtdb.csv")


rows with J or *: 4
rows with J or *: 0
rows with J or *: 0


In [4]:
y_train = list(train['label'])
y_val = list(val['label'])
y_test = list(test['label'])

In [6]:
X_train_esm = src.esm_extract(train)
X_val_esm = src.esm_extract(val)
X_test_esm = src.esm_extract(test)

Using cache found in /home/sjchoi/.cache/torch/hub/facebookresearch_esm_main
100%|██████████| 23067/23067 [22:51<00:00, 16.81it/s]
Using cache found in /home/sjchoi/.cache/torch/hub/facebookresearch_esm_main
100%|██████████| 2883/2883 [02:45<00:00, 17.45it/s]
Using cache found in /home/sjchoi/.cache/torch/hub/facebookresearch_esm_main
100%|██████████| 2884/2884 [02:48<00:00, 17.14it/s]


In [9]:
from transformers import AutoTokenizer, AutoModel

def bert_extract(df):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
    model = AutoModel.from_pretrained("google-bert/bert-base-uncased").to(device)
    model.eval()
    txt_emb = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Extracting"):
        desc = str(row.get('desc_nodup', '')).strip()
        name = str(row.get('name', '')).strip()
        rank = str(row.get('rank', '')).strip()
        combined_text = f"{desc}|{name}|{rank}"
        embedding = src.get_embedding(combined_text, tokenizer, model, device)
        txt_emb.append(embedding.numpy().tolist())
    
    return np.array(txt_emb)

In [10]:
X_train_text = bert_extract(train)
X_val_text   = bert_extract(val)
X_test_text  = bert_extract(test)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1093.13it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1191.65it/s, Materializing param=pooler.dense.weight]              

In [11]:
X_train_all = np.concatenate([X_train_esm, X_train_text], axis=1)
X_val_all   = np.concatenate([X_val_esm, X_val_text], axis=1)
X_test_all  = np.concatenate([X_test_esm, X_test_text], axis=1)

In [12]:
X_train_all.shape

(23067, 2048)

In [13]:
_X_train = np.concatenate([X_train_all, X_val_all], axis=0)
_y_train = np.concatenate([y_train, y_val], axis=0)

In [14]:
import warnings
warnings.filterwarnings(action='ignore', category=FutureWarning, module='xgboost')

from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, matthews_corrcoef, confusion_matrix

# Define the parameter grid to search
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.7, 1.0],
    'colsample_bytree': [0.7, 1.0]
}

# Initialize XGBClassifier
xgb = XGBClassifier(
    tree_method="hist",
    eval_metric="auc",
    random_state=42,
)

# Grid search with 5-fold cross-validation
grid_search = GridSearchCV(estimator=xgb, param_grid=param_grid, cv=5, scoring='roc_auc', verbose=1, n_jobs=-1, error_score="raise")

# Fit grid search
grid_search.fit(_X_train, _y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best cross-validation AUC: ", grid_search.best_score_)

# Evaluate on test data using best estimator
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_all)
y_proba = best_model.predict_proba(X_test_all)[:, 1]

Fitting 5 folds for each of 108 candidates, totalling 540 fits


/home/sjchoi/miniconda3/envs/node09/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best parameters found:  {'colsample_bytree': 0.7, 'learning_rate': 0.2, 'max_depth': 7, 'n_estimators': 200, 'subsample': 1.0}
Best cross-validation AUC:  0.9467866988910792


In [15]:
print("Test ROC AUC:", roc_auc_score(y_test, y_proba))
print("Classification Report:\n", classification_report(y_test, y_pred))

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print(f"Accuracy: {accuracy:.3f}")
print(f"F1-score: {f1:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"MCC: {mcc:.3f}")

Test ROC AUC: 0.9513618106305582
Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.90      0.89      1442
           1       0.90      0.87      0.89      1442

    accuracy                           0.89      2884
   macro avg       0.89      0.89      0.89      2884
weighted avg       0.89      0.89      0.89      2884

Accuracy: 0.887
F1-score: 0.885
Precision: 0.897
Recall: 0.873
MCC: 0.774


Best parameters found:  {'colsample_bytree': 0.7, 'learning_rate': 0.2, 'max_depth': 7, 'n_estimators': 200, 'subsample': 1.0}

In [16]:
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, matthews_corrcoef, confusion_matrix

best_params = {
    'colsample_bytree': 0.7,
    'learning_rate': 0.2,
    'max_depth': 7,
    'n_estimators': 200,
    'subsample': 1.0,
}

from xgboost import XGBClassifier

best_model = XGBClassifier(
    **best_params,
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
)

best_model.fit(_X_train, _y_train)
y_pred = best_model.predict(X_test_all)
y_proba = best_model.predict_proba(X_test_all)[:, 1]

print("Test ROC AUC:", roc_auc_score(y_test, y_proba))
print("Classification Report:\n", classification_report(y_test, y_pred))

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print(f"Accuracy: {accuracy:.3f}")
print(f"F1-score: {f1:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"MCC: {mcc:.3f}")
tn, fp, fn, tp = (confusion_matrix(y_test, y_pred)).ravel().tolist()
print("specificity", tn/(tn+fp))

Test ROC AUC: 0.9513618106305582
Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.90      0.89      1442
           1       0.90      0.87      0.89      1442

    accuracy                           0.89      2884
   macro avg       0.89      0.89      0.89      2884
weighted avg       0.89      0.89      0.89      2884

Accuracy: 0.887
F1-score: 0.885
Precision: 0.897
Recall: 0.873
MCC: 0.774
specificity 0.9001386962552012


In [17]:
print("Test ROC AUC:", roc_auc_score(y_test, y_proba))
print("Classification Report:\n", classification_report(y_test, y_pred))

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print(f"Accuracy: {accuracy:.3f}")
print(f"F1-score: {f1:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"MCC: {mcc:.3f}")

Test ROC AUC: 0.9513618106305582
Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.90      0.89      1442
           1       0.90      0.87      0.89      1442

    accuracy                           0.89      2884
   macro avg       0.89      0.89      0.89      2884
weighted avg       0.89      0.89      0.89      2884

Accuracy: 0.887
F1-score: 0.885
Precision: 0.897
Recall: 0.873
MCC: 0.774


In [18]:
import joblib

# After training and grid search as before
best_model = grid_search.best_estimator_

# Save the model to a file
joblib.dump(best_model, 'VFTextSeq_model.pkl')

# Later, you can load it back with
# loaded_model = joblib.load('best_xgb_model.pkl')


['VFTextSeq_model.pkl']

In [19]:
import pandas as pd
import joblib

# Load the trained model
loaded_model = joblib.load('VFTextSeq_model.pkl')


labels = list(test['label'])

# Predict
probabilities = loaded_model.predict_proba(X_test_all)[:,1]
predictions = loaded_model.predict(X_test_all)


# Prepare result dataframe with id, predicted label, and original label
result_df = pd.DataFrame({
    'id': test['id'],
    'prob': probabilities,
    'pred': predictions,
    'label': labels
})

# Save results
result_df.to_csv('VFTextSeq_test_prediction.csv', index=False)
